# 09 FS-Safe — Feature-Selected Regime-Aware Online Hybrid Ensemble

This version retains the original online ensemble as a mandatory incumbent and
adds a **historical-only, conservative market-state feature selection branch**.

Safeguards:
- Exclude every development prediction whose target date touches 2024–2025.
- Use only point-in-time features already defined by Notebook 02.
- Select regime features using resolved historical data from before each year.
- Compare Mutual Information, L1/Lasso and Spearman-ranked features, preceded
  by missingness, zero-variance and high-correlation filters.
- Use selected features for similarity-weighted *past expert losses*; never use
  the current or future actual outcome when selecting the current forecast.
- Keep the original Notebook 09 prediction unless an alternative has beaten
  it by a prespecified historical margin on already-resolved observations.
- Recompute all benchmark metrics after holdout-boundary filtering.

**No promised accuracy.** Results are unknown until this modified notebook
is executed on the user's project prediction files. Past notebook outputs are
not valid results of this revised version.

**Upstream caveat:** The statistical and ML prediction inputs must themselves
have been generated by leakage-safe historical tuning. This notebook audits
their alignment but cannot retroactively repair their training procedures.


In [ ]:
# ============================================================
# 1. IMPORTS AND CONFIGURATION
# ============================================================

from pathlib import Path
import warnings
import time
import math

import numpy as np
import pandas as pd

from scipy.stats import t as student_t

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 230)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

CURRENCIES = ["USD", "GBP", "EUR", "JPY"]
HORIZONS = [1, 3, 7]
YEARS = [2019, 2020, 2021, 2022, 2023]

MIN_HISTORY = 20

# Candidate online-hybrid profiles.
# These are intentionally compact to keep runtime reasonable.
PROFILE_GRID = []

for lookback in [40, 80, 160]:
    for eta in [0.25, 0.75, 1.50]:
        for shrink in [0.25, 0.50, 0.75, 1.00]:
            PROFILE_GRID.append({
                "lookback": lookback,
                "eta": eta,
                "shrink": shrink,
                "bias_window": min(40, lookback),
                "rw_guard": True,
            })

print("Candidate online profiles:", len(PROFILE_GRID))

In [ ]:
# ============================================================
# 2. PROJECT PATHS
# ============================================================

def find_root():
    for root in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        rw = root/"outputs"/"predictions"/"04_random_walk_predictions_2019_2023.csv"
        stat = root/"outputs"/"predictions"/"05_statistical_predictions_2019_2023.csv"

        if rw.exists() and stat.exists():
            return root.resolve()

    raise FileNotFoundError(
        "Could not locate Script 04 and Script 05 prediction outputs."
    )

ROOT = find_root()

PRED_DIR = ROOT/"outputs"/"predictions"
TABLE_DIR = ROOT/"outputs"/"tables"

TABLE_DIR.mkdir(parents=True, exist_ok=True)

RW_PATH = PRED_DIR/"04_random_walk_predictions_2019_2023.csv"
STAT_PATH = PRED_DIR/"05_statistical_predictions_2019_2023.csv"

ML_PATHS = [
    PRED_DIR/"06_FAST_ml_predictions_2019_2023.csv",
    PRED_DIR/"06_ml_predictions_2019_2023.csv",
]

ML_PATH = next((p for p in ML_PATHS if p.exists()), None)

if ML_PATH is None:
    raise FileNotFoundError("No Script 06 ML prediction file found.")

HYBRID07_PATH = PRED_DIR/"07_hybrid_predictions_2019_2023.csv"

print("RW:", RW_PATH.name)
print("STAT:", STAT_PATH.name)
print("ML:", ML_PATH.name)
print("Hybrid07 available:", HYBRID07_PATH.exists())

In [ ]:
# ============================================================
# 3. ROBUST COLUMN DETECTION
# ============================================================

def pick_column(frame, candidates, label):
    for c in candidates:
        if c in frame.columns:
            return c

    raise ValueError(
        f"Could not find {label}. Available columns: {list(frame.columns)}"
    )


rw = pd.read_csv(
    RW_PATH,
    parse_dates=["Forecast_Origin", "Target_Date"]
)

stat = pd.read_csv(
    STAT_PATH,
    parse_dates=["Forecast_Origin", "Target_Date"]
)

ml = pd.read_csv(
    ML_PATH,
    parse_dates=["Forecast_Origin", "Target_Date"]
)

stat_pred_col = pick_column(
    stat,
    [
        "Stat_Prediction",
        "Statistical_Prediction",
        "Model_Prediction",
        "Prediction",
    ],
    "statistical prediction column"
)

ml_pred_col = pick_column(
    ml,
    [
        "ML_Prediction",
        "Model_Prediction",
        "Prediction",
    ],
    "ML prediction column"
)

print("Stat column:", stat_pred_col)
print("ML column:", ml_pred_col)

In [ ]:
# ============================================================
# 4. BUILD MATCHED FORECAST PANEL
# ============================================================

KEYS = [
    "Currency",
    "Horizon",
    "Forecast_Origin",
]

base = rw[
    KEYS
    + [
        "Target_Date",
        "Actual",
        "RW_Prediction",
    ]
].copy()

base = base.merge(
    stat[
        KEYS
        + [stat_pred_col]
    ].rename(
        columns={
            stat_pred_col: "STAT"
        }
    ),
    on=KEYS,
    how="inner"
)

base = base.merge(
    ml[
        KEYS
        + [ml_pred_col]
    ].rename(
        columns={
            ml_pred_col: "ML"
        }
    ),
    on=KEYS,
    how="inner"
)

base = base.rename(
    columns={
        "RW_Prediction": "RW"
    }
)

# Optional Script 07 hybrid expert.
if HYBRID07_PATH.exists():

    hy = pd.read_csv(
        HYBRID07_PATH,
        parse_dates=["Forecast_Origin", "Target_Date"]
    )

    hy_col = pick_column(
        hy,
        [
            "Hybrid_Prediction",
            "Ensemble_Prediction",
            "Prediction",
        ],
        "Script 07 hybrid prediction column"
    )

    base = base.merge(
        hy[
            KEYS
            + [hy_col]
        ].rename(
            columns={
                hy_col: "HYBRID07"
            }
        ),
        on=KEYS,
        how="left"
    )

base["Validation_Year"] = (
    base["Forecast_Origin"].dt.year
)

base = (
    base
    .sort_values(
        [
            "Currency",
            "Horizon",
            "Forecast_Origin",
        ]
    )
    .reset_index(drop=True)
)

assert base["Forecast_Origin"].max() < pd.Timestamp("2024-01-01")

print("Matched rows:", len(base))
display(base.head())

In [ ]:
# STRICT RESEARCH BOUNDARY AND DATA-INTEGRITY AUDITS
HOLDOUT_START = pd.Timestamp('2024-01-01')
for label, source_frame in [('RW', rw), ('STAT', stat), ('ML', ml)]:
    if source_frame.duplicated(KEYS).any():
        raise ValueError(f'{label}: duplicate currency/horizon/origin keys')
    if 'Actual' in source_frame.columns:
        matched = rw[KEYS + ['Actual']].merge(
            source_frame[KEYS + ['Actual']], on=KEYS,
            suffixes=('_rw', '_source'), how='inner')
        if not np.allclose(matched['Actual_rw'], matched['Actual_source'],
                           rtol=0, atol=1e-8, equal_nan=True):
            raise ValueError(f'{label}: target values disagree with Random Walk')

if base.duplicated(KEYS).any():
    raise ValueError('Merged panel has duplicate forecast keys')
if (base['Target_Date'] <= base['Forecast_Origin']).any():
    raise ValueError('Forecast target dates must follow their origins')

removed_boundary = int((base['Target_Date'] >= HOLDOUT_START).sum())
base = base.loc[
    (base['Forecast_Origin'] < HOLDOUT_START)
    & (base['Target_Date'] < HOLDOUT_START)
    & base['Actual'].notna()
].copy().reset_index(drop=True)
print('Excluded observations whose labels enter final holdout:', removed_boundary)
print('Safe development prediction rows:', len(base))
assert (base['Target_Date'] < HOLDOUT_START).all()

# 5. Create hybrid expert forecasts

We add conservative combinations as additional experts.

These are **not fitted on the evaluation outcomes**. They are predefined forecast combinations:

- `RW_STAT_25`: 75% RW + 25% Statistical
- `RW_STAT_50`: 50% RW + 50% Statistical
- `RW_ML_25`: 75% RW + 25% ML
- `RW_ML_50`: 50% RW + 50% ML
- `STAT_ML_50`: 50% Statistical + 50% ML

The online learner decides whether any of them deserve weight.

In [ ]:
# ============================================================
# 5. EXPERT LIBRARY
# ============================================================

base["RW_STAT_25"] = (
    0.75 * base["RW"]
    + 0.25 * base["STAT"]
)

base["RW_STAT_50"] = (
    0.50 * base["RW"]
    + 0.50 * base["STAT"]
)

base["RW_ML_25"] = (
    0.75 * base["RW"]
    + 0.25 * base["ML"]
)

base["RW_ML_50"] = (
    0.50 * base["RW"]
    + 0.50 * base["ML"]
)

base["STAT_ML_50"] = (
    0.50 * base["STAT"]
    + 0.50 * base["ML"]
)

EXPERTS = [
    "RW",
    "STAT",
    "ML",
    "RW_STAT_25",
    "RW_STAT_50",
    "RW_ML_25",
    "RW_ML_50",
    "STAT_ML_50",
]

if "HYBRID07" in base.columns:
    EXPERTS.append("HYBRID07")

print("Expert pool:")
print(EXPERTS)

In [ ]:
for expert in EXPERTS:
    base[expert] = pd.to_numeric(base[expert], errors='coerce')
    bad = ~np.isfinite(base[expert].to_numpy(dtype=float))
    if bad.any():
        print(f'{expert}: {bad.sum()} unavailable forecasts -> RW fallback')
        base.loc[bad, expert] = base.loc[bad, 'RW']
if not np.isfinite(base[['Actual'] + EXPERTS].to_numpy(dtype=float)).all():
    raise ValueError('Nonfinite labels or expert forecasts after fallback')

In [ ]:
# ============================================================
# 6. METRICS
# ============================================================

def rmse(y, p):
    y = np.asarray(y, float)
    p = np.asarray(p, float)

    m = (
        np.isfinite(y)
        & np.isfinite(p)
    )

    if m.sum() == 0:
        return np.nan

    return float(
        np.sqrt(
            np.mean(
                (y[m]-p[m])**2
            )
        )
    )


def mae(y, p):
    y = np.asarray(y, float)
    p = np.asarray(p, float)

    m = (
        np.isfinite(y)
        & np.isfinite(p)
    )

    if m.sum() == 0:
        return np.nan

    return float(
        np.mean(
            np.abs(
                y[m]-p[m]
            )
        )
    )


def skill(m, r):
    if (
        not np.isfinite(m)
        or not np.isfinite(r)
        or r == 0
    ):
        return np.nan

    return float(
        1 - m/r
    )


def theils_u(m, r):
    if (
        not np.isfinite(m)
        or not np.isfinite(r)
        or r == 0
    ):
        return np.nan

    return float(
        m/r
    )


def dm_test(y, model, rw_pred, horizon=1):
    y = np.asarray(y,float)
    model = np.asarray(model,float)
    rw_pred = np.asarray(rw_pred,float)

    mask = (
        np.isfinite(y)
        & np.isfinite(model)
        & np.isfinite(rw_pred)
    )

    y = y[mask]
    model = model[mask]
    rw_pred = rw_pred[mask]

    n = len(y)

    if n < 20:
        return np.nan, np.nan

    d = (
        (y-rw_pred)**2
        - (y-model)**2
    )

    lag = max(
        0,
        int(horizon)-1
    )

    lrv = np.var(
        d,
        ddof=1
    )

    for k in range(1,lag+1):

        if len(d) <= k:
            break

        cov = np.cov(
            d[k:],
            d[:-k],
            ddof=1
        )[0,1]

        lrv += (
            2
            * (1-k/(lag+1))
            * cov
        )

    if (
        not np.isfinite(lrv)
        or lrv <= 0
    ):
        return np.nan, np.nan

    stat = (
        np.mean(d)
        / np.sqrt(lrv/n)
    )

    p = (
        2
        * student_t.sf(
            abs(stat),
            df=n-1
        )
    )

    return float(stat), float(p)

# 7. Online bias correction

For each non-RW expert, the notebook estimates recent forecast bias using **resolved past observations only**.

\[
Bias_m = mean(y-\hat y_m)
\]

and:

\[
\hat y_m^{BC} = \hat y_m + Bias_m
\]

This can remove persistent under/over-prediction without looking forward.

In [ ]:
# ============================================================
# 7. ONLINE HELPERS
# ============================================================

def resolved_history(task, origin):
    return task[
        (task["Target_Date"] <= origin)
        & (task["Forecast_Origin"] < origin)
    ].copy()


def bias_corrected_current(
    current_row,
    history,
    expert,
    bias_window
):
    pred = float(
        current_row[expert]
    )

    if expert == "RW":
        return pred

    h = history.tail(
        bias_window
    )

    if len(h) < MIN_HISTORY:
        return pred

    errors = (
        h["Actual"]
        - h[expert]
    )

    errors = errors[
        np.isfinite(errors)
    ]

    if len(errors) < MIN_HISTORY:
        return pred

    bias = float(
        errors.mean()
    )

    return pred + bias

# 8. Exponentially weighted expert losses

Recent errors receive more importance.

We normalize each expert's loss by RW loss, so the online learner focuses on **performance relative to Random Walk**.

In [ ]:
# ============================================================
# 8. EXPERT WEIGHTS
# ============================================================

def online_weights(
    history,
    profile
):
    if len(history) < MIN_HISTORY:
        return {
            e: (
                1.0
                if e == "RW"
                else 0.0
            )
            for e in EXPERTS
        }

    h = history.tail(
        int(profile["lookback"])
    ).copy()

    n = len(h)

    # More weight for recent rows.
    ages = np.arange(
        n-1,
        -1,
        -1
    )

    decay = np.exp(
        -ages
        / max(
            10.0,
            profile["lookback"]/3
        )
    )

    losses = {}

    for e in EXPERTS:

        err2 = (
            h["Actual"].to_numpy(float)
            - h[e].to_numpy(float)
        ) ** 2

        valid = np.isfinite(err2)

        if valid.sum() < MIN_HISTORY:
            losses[e] = np.inf
            continue

        w = decay[valid]

        losses[e] = float(
            np.sum(
                w * err2[valid]
            )
            / np.sum(w)
        )

    rw_loss = losses.get(
        "RW",
        np.inf
    )

    if (
        not np.isfinite(rw_loss)
        or rw_loss <= 0
    ):
        return {
            e: (
                1.0
                if e == "RW"
                else 0.0
            )
            for e in EXPERTS
        }

    raw = {}

    for e in EXPERTS:

        loss_e = losses[e]

        if not np.isfinite(loss_e):
            raw[e] = 0.0
            continue

        relative = (
            loss_e / rw_loss
        )

        # Penalize experts worse than RW exponentially.
        raw[e] = math.exp(
            -profile["eta"]
            * relative
        )

        # Optional guard:
        # strongly downweight experts clearly worse than RW.
        if (
            profile["rw_guard"]
            and relative > 1.05
        ):
            raw[e] *= 0.10

    # Give RW a stable floor.
    raw["RW"] = max(
        raw.get("RW",0.0),
        1.0
    )

    total = sum(
        raw.values()
    )

    if total <= 0:
        return {
            e: (
                1.0
                if e == "RW"
                else 0.0
            )
            for e in EXPERTS
        }

    return {
        e: raw[e]/total
        for e in EXPERTS
    }

# 9. Online hybrid simulation

The ensemble is generated sequentially.

No forecast can use an unresolved future outcome.

In [ ]:
# ============================================================
# 9. SIMULATE ONE ONLINE PROFILE
# ============================================================

def simulate_profile(
    task,
    profile,
    profile_id
):
    task = (
        task
        .sort_values(
            "Forecast_Origin"
        )
        .reset_index(drop=True)
    )

    rows = []

    for _, row in task.iterrows():

        origin = row[
            "Forecast_Origin"
        ]

        history = resolved_history(
            task,
            origin
        )

        weights = online_weights(
            history,
            profile
        )

        expert_preds = {}

        for e in EXPERTS:

            expert_preds[e] = (
                bias_corrected_current(
                    row,
                    history,
                    e,
                    int(
                        profile[
                            "bias_window"
                        ]
                    )
                )
            )

        ensemble = sum(
            weights[e]
            * expert_preds[e]
            for e in EXPERTS
        )

        # Final conservative shrinkage toward RW.
        rw_pred = float(
            row["RW"]
        )

        final = (
            rw_pred
            + profile["shrink"]
            * (
                ensemble
                - rw_pred
            )
        )

        out = {
            "Currency": row["Currency"],
            "Horizon": row["Horizon"],
            "Forecast_Origin": origin,
            "Target_Date": row["Target_Date"],
            "Validation_Year": row["Validation_Year"],
            "Actual": row["Actual"],
            "RW_Prediction": rw_pred,
            "Online_Hybrid_Prediction": final,
            "Profile_ID": profile_id,
            "Lookback": profile["lookback"],
            "Eta": profile["eta"],
            "Shrink": profile["shrink"],
            "Bias_Window": profile["bias_window"],
            "Resolved_History": len(history),
        }

        for e in EXPERTS:
            out[f"Weight_{e}"] = weights[e]

        rows.append(
            out
        )

    return pd.DataFrame(
        rows
    )

# 10. Simulate all candidate profiles

This is still much faster than refitting ML models because all underlying model forecasts already exist.

In [ ]:
# ============================================================
# 10. SIMULATE CANDIDATE PROFILES
# ============================================================

all_profiles = []

start_all = time.time()

for cur in CURRENCIES:

    for h in HORIZONS:

        task = base[
            (base["Currency"] == cur)
            & (base["Horizon"] == h)
        ].copy()

        print(
            f"Simulating {cur} H{h}...",
            flush=True
        )

        for pid, profile in enumerate(
            PROFILE_GRID
        ):

            sim = simulate_profile(
                task,
                profile,
                pid
            )

            all_profiles.append(
                sim
            )

profile_predictions = pd.concat(
    all_profiles,
    ignore_index=True
)

print(
    f"Simulation finished in "
    f"{(time.time()-start_all)/60:.2f} minutes."
)

print(
    "Rows:",
    len(profile_predictions)
)

# 11. Choose profile for each outer year using earlier years only

This is the key anti-overfitting step.

- 2019 uses a predefined conservative profile;
- 2020 profile is chosen from 2019 only;
- 2021 from 2019–2020;
- 2022 from 2019–2021;
- 2023 from 2019–2022.

The evaluated year never chooses its own profile.

In [ ]:
# ============================================================
# 11. HISTORICAL PROFILE SELECTION
# ============================================================

# Conservative default profile:
DEFAULT_PROFILE_ID = 0

selection_rows = []
final_frames = []

for cur in CURRENCIES:

    for h in HORIZONS:

        task_profiles = profile_predictions[
            (profile_predictions["Currency"] == cur)
            & (profile_predictions["Horizon"] == h)
        ].copy()

        for year in YEARS:

            if year == YEARS[0]:

                chosen_id = DEFAULT_PROFILE_ID
                historical_rmse = np.nan

            else:

                prior = task_profiles[
                    task_profiles[
                        "Validation_Year"
                    ] < year
                ].copy()

                score_rows = []

                for pid, g in prior.groupby(
                    "Profile_ID"
                ):

                    score_rows.append({
                        "Profile_ID": pid,
                        "Historical_RMSE": rmse(
                            g["Actual"],
                            g["Online_Hybrid_Prediction"]
                        )
                    })

                score_df = (
                    pd.DataFrame(
                        score_rows
                    )
                    .sort_values(
                        [
                            "Historical_RMSE",
                            "Profile_ID",
                        ]
                    )
                )

                best = score_df.iloc[0]

                chosen_id = int(
                    best["Profile_ID"]
                )

                historical_rmse = float(
                    best[
                        "Historical_RMSE"
                    ]
                )

            chosen = task_profiles[
                (task_profiles["Validation_Year"] == year)
                & (task_profiles["Profile_ID"] == chosen_id)
            ].copy()

            chosen[
                "Selected_Profile_ID"
            ] = chosen_id

            final_frames.append(
                chosen
            )

            p = PROFILE_GRID[
                chosen_id
            ]

            selection_rows.append({
                "Currency": cur,
                "Horizon": h,
                "Outer_Year": year,
                "Selected_Profile_ID": chosen_id,
                "Lookback": p["lookback"],
                "Eta": p["eta"],
                "Shrink": p["shrink"],
                "Bias_Window": p["bias_window"],
                "Historical_Selection_RMSE": historical_rmse,
            })

final_predictions = pd.concat(
    final_frames,
    ignore_index=True
)

profile_selections = pd.DataFrame(
    selection_rows
)

display(
    profile_selections
)

# 12. Yearly results

In [ ]:
# ============================================================
# 12. YEARLY RESULTS
# ============================================================

yearly = []

for (
    cur,
    h,
    year
), g in final_predictions.groupby(
    [
        "Currency",
        "Horizon",
        "Validation_Year",
    ]
):

    mr = rmse(
        g["Actual"],
        g["Online_Hybrid_Prediction"]
    )

    ma = mae(
        g["Actual"],
        g["Online_Hybrid_Prediction"]
    )

    rr = rmse(
        g["Actual"],
        g["RW_Prediction"]
    )

    _, p = dm_test(
        g["Actual"],
        g["Online_Hybrid_Prediction"],
        g["RW_Prediction"],
        h
    )

    yearly.append({
        "Currency": cur,
        "Horizon": int(h),
        "Validation_Year": int(year),
        "Observations": len(g),
        "RMSE": mr,
        "MAE": ma,
        "Forecast_Skill_vs_RW": skill(
            mr,
            rr
        ),
        "Theils_U": theils_u(
            mr,
            rr
        ),
        "DM_pvalue_vs_RW": p,
    })

yearly_results = pd.DataFrame(
    yearly
).sort_values(
    [
        "Currency",
        "Horizon",
        "Validation_Year",
    ]
)

display(
    yearly_results
)

yearly_results.to_csv(
    TABLE_DIR/"09FS_safe_baseline_online_hybrid_yearly_results.csv",
    index=False
)

# 13. Pooled 2019–2023 results

In [ ]:
# ============================================================
# 13. POOLED RESULTS
# ============================================================

pooled = []

for (
    cur,
    h
), g in final_predictions.groupby(
    [
        "Currency",
        "Horizon",
    ]
):

    mr = rmse(
        g["Actual"],
        g["Online_Hybrid_Prediction"]
    )

    ma = mae(
        g["Actual"],
        g["Online_Hybrid_Prediction"]
    )

    rr = rmse(
        g["Actual"],
        g["RW_Prediction"]
    )

    _, p = dm_test(
        g["Actual"],
        g["Online_Hybrid_Prediction"],
        g["RW_Prediction"],
        h
    )

    pooled.append({
        "Currency": cur,
        "Horizon": int(h),
        "Observations": len(g),
        "RMSE": mr,
        "MAE": ma,
        "Forecast_Skill_vs_RW": skill(
            mr,
            rr
        ),
        "Theils_U": theils_u(
            mr,
            rr
        ),
        "DM_pvalue_vs_RW": p,
    })

pooled_results = pd.DataFrame(
    pooled
).sort_values(
    [
        "Currency",
        "Horizon",
    ]
)

display(
    pooled_results
)

pooled_results.to_csv(
    TABLE_DIR/"09FS_safe_baseline_online_hybrid_pooled_results.csv",
    index=False
)

In [ ]:
# ============================================================
# 14. WIN SUMMARY
# ============================================================

wins = pooled_results.copy()

wins["Beats_RW"] = (
    wins["Forecast_Skill_vs_RW"] > 0
)

wins["DM_Significant_5pct"] = (
    wins["DM_pvalue_vs_RW"] < 0.05
)

display(
    wins
)

print(
    "Tasks beating RW:",
    int(
        wins[
            "Beats_RW"
        ].sum()
    ),
    "/",
    len(wins)
)

print(
    "Positive skill + DM p<0.05:",
    int(
        (
            wins["Beats_RW"]
            & wins["DM_Significant_5pct"]
        ).sum()
    ),
    "/",
    len(wins)
)

# 15. Expert-weight summary

This tells us which models the online hybrid actually trusted.

In [ ]:
# ============================================================
# 15. EXPERT WEIGHT SUMMARY
# ============================================================

weight_cols = [
    f"Weight_{e}"
    for e in EXPERTS
]

agg_dict = {
    c: "mean"
    for c in weight_cols
}

weight_summary = (
    final_predictions
    .groupby(
        [
            "Currency",
            "Horizon",
        ]
    )
    .agg(
        agg_dict
    )
    .reset_index()
)

display(
    weight_summary
)

weight_summary.to_csv(
    TABLE_DIR/"09FS_safe_baseline_expert_weight_summary.csv",
    index=False
)

# 16. Save prediction-level evidence

In [ ]:
# ============================================================
# 16. SAVE OUTPUTS
# ============================================================

final_predictions.to_csv(
    PRED_DIR/"09FS_safe_baseline_online_hybrid_predictions_2019_2023.csv",
    index=False
)

profile_selections.to_csv(
    TABLE_DIR/"09FS_safe_baseline_online_hybrid_profile_selections.csv",
    index=False
)

print("Saved Script 09 outputs.")

# 17. Quality gate

In [ ]:
# ============================================================
# 17. QUALITY GATE
# ============================================================

base_keys = set(
    zip(
        base["Currency"],
        base["Horizon"],
        base["Forecast_Origin"],
    )
)

final_keys = set(
    zip(
        final_predictions["Currency"],
        final_predictions["Horizon"],
        final_predictions["Forecast_Origin"],
    )
)

quality_gate = pd.DataFrame([
    {
        "Area": "Holdout isolation",
        "Status": "PASS",
        "Evidence": "Only 2019–2023 development prediction files loaded"
    },
    {
        "Area": "Hybrid model pool",
        "Status": "PASS",
        "Evidence": "RW + Statistical + ML + predefined blends + optional Script07"
    },
    {
        "Area": "Resolved-error rule",
        "Status": "PASS",
        "Evidence": "Online weights use only outcomes already known at forecast origin"
    },
    {
        "Area": "Bias correction",
        "Status": "PASS",
        "Evidence": "Rolling bias uses resolved past errors only"
    },
    {
        "Area": "Outer-year profile selection",
        "Status": "PASS",
        "Evidence": "Profile selected using earlier years only"
    },
    {
        "Area": "Matched origins",
        "Status": (
            "PASS"
            if base_keys == final_keys
            else "REVIEW"
        ),
        "Evidence": (
            f"Base={len(base_keys)}, Final={len(final_keys)}"
        )
    },
])

display(
    quality_gate
)

quality_gate.to_csv(
    TABLE_DIR/"09FS_safe_baseline_quality_gate.csv",
    index=False
)

# 18. Final interpretation

A task is a numerical benchmark win when:

\[
ForecastSkill>0
\]

and:

\[
Theil's\ U<1
\]

A DM p-value below 0.05 strengthens the evidence only when the skill is positive.

After Script 09, the development design should be frozen rather than repeatedly changed to chase the pooled score.

In [ ]:
print("="*84)
print("SCRIPT 09 — REGIME-AWARE ONLINE HYBRID ENSEMBLE COMPLETE")
print("="*84)

print("\nPOOLED RESULTS:")
display(pooled_results)

print("\nPROFILE SELECTIONS:")
display(profile_selections)

print("\nAVERAGE EXPERT WEIGHTS:")
display(weight_summary)

print(
    "\nTasks beating RW:",
    int(
        (
            pooled_results[
                "Forecast_Skill_vs_RW"
            ] > 0
        ).sum()
    ),
    "/",
    len(pooled_results)
)

print("\n2024–2025 remains untouched.")

## 19. Point-in-time market-state feature selection

This experiment does **not** retrain the existing statistical or ML experts.
It tests whether market-state similarities can improve how the online ensemble
weights their already-generated forecasts. The selection label is the *realized
absolute Random Walk error*, observed only when its target date has resolved.

We compare unmodified Notebook 09 (BASE) with three low-complexity selectors:
Mutual Information (MI), L1/Lasso, and absolute Spearman association.
Each selector is preceded by training-only missingness, variance and correlation
filters. This is feature **selection**, not creation of new future-known data.

In [ ]:
import json
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_regression
from sklearn.linear_model import Lasso

DEV_PATH = ROOT/'data'/'processed'/'02_development_model_dataset.csv'
FEATURE_PATH = ROOT/'data'/'processed'/'02_feature_sets.json'
if not DEV_PATH.exists() or not FEATURE_PATH.exists():
    raise FileNotFoundError('Run Notebook 02: development data and feature definitions required')

dev = pd.read_csv(DEV_PATH, parse_dates=['Date'])
if dev['Date'].max() >= HOLDOUT_START:
    raise ValueError('Development feature source includes final holdout origins')
if dev['Date'].duplicated().any():
    raise ValueError('Duplicate feature dates')
with open(FEATURE_PATH, encoding='utf-8') as f:
    feature_sets = json.load(f)

FS_METHODS = ['BASE', 'MI', 'L1', 'SPEARMAN']
FS_TOP_K = 5
FS_MIN_TRAIN = 120
FS_MIN_VALID = 20
FS_IMPROVEMENT_MARGIN = 0.0025   # 0.25% historical RMSE improvement required
FS_MAX_YEARLY_DEGRADATION = 0.05 # no historical year >5% worse than BASE
FS_SIMILARITY_FLOOR = 0.15
FS_CORR_CUTOFF = 0.97
FS_RANDOM_STATE = 42

def safe_feature_name(c):
    forbidden = ['_target_H', '_change_H', '_pct_change_H', '_log_change_H']
    return c != 'Date' and not c.startswith('Target_Date_H') and not any(x in c for x in forbidden)

candidate_by_currency = {}
for cur in CURRENCIES:
    names = feature_sets[cur]['FULL_PIT']
    names = [c for c in dict.fromkeys(names) if c in dev.columns and safe_feature_name(c)]
    names = [c for c in names if pd.api.types.is_numeric_dtype(dev[c])]
    if not names:
        raise ValueError(f'{cur}: no safe numerical point-in-time features')
    candidate_by_currency[cur] = names

context_base = base.copy()
feature_union = list(dict.fromkeys(c for names in candidate_by_currency.values() for c in names))
context_base = context_base.merge(
    dev[['Date'] + feature_union].rename(columns={'Date': 'Forecast_Origin'}),
    on='Forecast_Origin', how='left', validate='many_to_one'
)
if len(context_base) != len(base):
    raise RuntimeError('Feature join unexpectedly changed the prediction count')

incumbent = final_predictions[KEYS + ['Online_Hybrid_Prediction']].rename(
    columns={'Online_Hybrid_Prediction': 'BASE_Prediction'})
context_base = context_base.merge(incumbent, on=KEYS, how='left', validate='one_to_one')
if context_base['BASE_Prediction'].isna().any():
    raise RuntimeError('Original Notebook 09 incumbent predictions are not fully aligned')
print('Historical market-state candidate features:',
      {cur: len(v) for cur, v in candidate_by_currency.items()})

In [ ]:
def select_context_features(task, cur, year, method):
    """Fit all selector preprocessing on *resolved pre-year* rows only."""
    start = pd.Timestamp(f'{year}-01-01')
    prior = task[(task['Forecast_Origin'] < start) & (task['Target_Date'] < start)]
    audit = {'Currency': cur, 'Horizon': int(task['Horizon'].iloc[0]),
             'Outer_Year': year, 'Method': method, 'Training_Rows': len(prior),
             'Max_Selector_Target_Date': prior['Target_Date'].max(),
             'Selected_Features': '', 'Selected_Count': 0}
    if method == 'BASE' or len(prior) < FS_MIN_TRAIN:
        return None, audit

    candidates = candidate_by_currency[cur]
    X0 = prior[candidates].replace([np.inf, -np.inf], np.nan)
    usable = [c for c in candidates if X0[c].notna().mean() >= 0.80
              and X0[c].nunique(dropna=True) > 5]
    if len(usable) == 0:
        return None, audit
    X = X0[usable]
    # Keep only one of nearly duplicate features, according to history alone.
    corr = X.corr(method='spearman').abs().fillna(0)
    keep = []
    for c in usable:
        if all(corr.loc[c, old] < FS_CORR_CUTOFF for old in keep):
            keep.append(c)
    if not keep:
        return None, audit

    imputer = SimpleImputer(strategy='median', keep_empty_features=True)
    Ximp = imputer.fit_transform(X[keep])
    scaler = StandardScaler()
    Xstd = scaler.fit_transform(Ximp)
    # Horizon-normalized RW absolute error is known for prior resolved labels.
    y = np.abs(prior['Actual'].to_numpy(float) - prior['RW'].to_numpy(float))
    y /= np.maximum(np.abs(prior['RW'].to_numpy(float)), 1e-8)
    high = np.nanquantile(y, 0.99)
    y = np.log1p(np.clip(y, 0, high) * 1000.0)
    if not np.isfinite(y).all() or np.std(y) <= 1e-12:
        return None, audit

    if method == 'MI':
        scores = mutual_info_regression(Xstd, y, n_neighbors=3,
                                        random_state=FS_RANDOM_STATE)
    elif method == 'SPEARMAN':
        scores = np.array([abs(pd.Series(Xstd[:, j]).corr(
            pd.Series(y), method='spearman')) for j in range(Xstd.shape[1])])
    elif method == 'L1':
        ystd = (y-y.mean()) / max(y.std(), 1e-8)
        lasso = Lasso(alpha=0.05, max_iter=10000, random_state=FS_RANDOM_STATE)
        lasso.fit(Xstd, ystd)
        scores = np.abs(lasso.coef_)
    else:
        raise ValueError(f'Unknown selection method: {method}')
    scores = np.nan_to_num(scores, nan=0.0, posinf=0.0, neginf=0.0)
    if scores.max() <= 1e-12:
        return None, audit
    order = np.argsort(-scores, kind='stable')[:FS_TOP_K]
    chosen = [keep[j] for j in order]
    # Refit selected scaler/imputer strictly on the same historical rows.
    selected_imputer = SimpleImputer(strategy='median', keep_empty_features=True)
    selected_scaler = StandardScaler()
    selected_scaler.fit(selected_imputer.fit_transform(prior[chosen].replace(
        [np.inf, -np.inf], np.nan)))
    Xall = selected_scaler.transform(selected_imputer.transform(task[chosen].replace(
        [np.inf, -np.inf], np.nan)))
    Xall = np.clip(np.nan_to_num(Xall, nan=0.0, posinf=0.0, neginf=0.0), -6, 6)
    audit['Selected_Features'] = '|'.join(chosen)
    audit['Selected_Count'] = len(chosen)
    return Xall, audit

## 20. Regime-similar resolved errors and conservative model choice

For a forecast at time t, only prior origins whose target dates are at or
before t can enter expert-loss calculations. Selected state features modestly
reweight those errors by similarity to the current state. A candidate method is
used only when its *earlier-year* results outperform the unmodified ensemble
by at least 0.25%, without a major loss in any earlier year.

The original ensemble remains an explicit candidate and the default.

In [ ]:
def contextual_weights(resolved, state_rows, current_state, profile):
    """Weights from recency AND market similarity, without future outcomes."""
    if len(resolved) < MIN_HISTORY:
        return None
    h = resolved.tail(int(profile['lookback']))
    recent_x = state_rows[h.index.to_numpy()]
    dist2 = np.mean((recent_x - current_state) ** 2, axis=1)
    similarity = np.clip(np.exp(-0.5*np.minimum(dist2, 40)),
                         FS_SIMILARITY_FLOOR, 1.0)
    ages = np.arange(len(h)-1, -1, -1)
    recency = np.exp(-ages / max(10.0, profile['lookback']/3.0))
    observation_weights = recency * similarity

    losses = {}
    for expert in EXPERTS:
        err2 = (h['Actual'].to_numpy(float) - h[expert].to_numpy(float)) ** 2
        valid = np.isfinite(err2)
        losses[expert] = (float(np.average(err2[valid],
                            weights=observation_weights[valid]))
                          if valid.sum() >= MIN_HISTORY else np.inf)
    rw_loss = losses.get('RW', np.inf)
    if not np.isfinite(rw_loss) or rw_loss <= 0:
        return None
    raw = {}
    for expert in EXPERTS:
        relative = losses[expert]/rw_loss
        raw[expert] = math.exp(-profile['eta'] * relative) if np.isfinite(relative) else 0.0
        if profile['rw_guard'] and relative > 1.05:
            raw[expert] *= 0.10
    raw['RW'] = max(raw['RW'], 1.0)
    denominator = sum(raw.values())
    return {expert: raw[expert]/denominator for expert in EXPERTS} if denominator > 0 else None

def simulate_contextual_year(task, year, profile, transformed):
    """One historical-only selected-feature experiment for a given outer year."""
    rows = []
    year_rows = task.index[task['Validation_Year'].eq(year)]
    for pos in year_rows:
        row = task.iloc[pos]
        origin = row['Forecast_Origin']
        resolved = task[(task['Forecast_Origin'] < origin)
                        & (task['Target_Date'] <= origin)]
        pred = float(row['BASE_Prediction'])
        if len(resolved) >= FS_MIN_VALID:
            weights = contextual_weights(resolved, transformed, transformed[pos], profile)
            if weights is not None:
                expert_predictions = {
                    e: bias_corrected_current(row, resolved, e,
                                              int(profile['bias_window']))
                    for e in EXPERTS
                }
                ensemble = sum(weights[e]*expert_predictions[e] for e in EXPERTS)
                pred = float(row['RW'] + profile['shrink']*(ensemble-row['RW']))
        rows.append({'Currency': row['Currency'], 'Horizon': int(row['Horizon']),
                     'Forecast_Origin': origin, 'Target_Date': row['Target_Date'],
                     'Validation_Year': year, 'Actual': float(row['Actual']),
                     'RW_Prediction': float(row['RW']),
                     'BASE_Prediction': float(row['BASE_Prediction']),
                     'Method_Prediction': pred})
    return pd.DataFrame(rows)

# Build each candidate using only selector labels strictly before its year.
# Historical candidate simulations are used only for later-year decisions.
simulations = []
fs_audit_rows = []
for cur in CURRENCIES:
    for h in HORIZONS:
        task = context_base[(context_base['Currency'] == cur)
                            & (context_base['Horizon'] == h)].copy()
        task = task.sort_values('Forecast_Origin').reset_index(drop=True)
        selected_profiles = profile_selections[(profile_selections['Currency'] == cur)
                                              & (profile_selections['Horizon'] == h)]
        for year in YEARS:
            selected = selected_profiles[selected_profiles['Outer_Year'] == year]
            if len(selected) != 1:
                raise RuntimeError(f'Cannot identify historical profile for {cur} H{h}, {year}')
            profile = PROFILE_GRID[int(selected.iloc[0]['Selected_Profile_ID'])]
            for method in FS_METHODS:
                if method == 'BASE':
                    year_df = task[task['Validation_Year'] == year]
                    sim = pd.DataFrame({
                        'Currency': cur, 'Horizon': h,
                        'Forecast_Origin': year_df['Forecast_Origin'],
                        'Target_Date': year_df['Target_Date'],
                        'Validation_Year': year,
                        'Actual': year_df['Actual'], 'RW_Prediction': year_df['RW'],
                        'BASE_Prediction': year_df['BASE_Prediction'],
                        'Method_Prediction': year_df['BASE_Prediction']})
                    audit = {'Currency': cur, 'Horizon': h, 'Outer_Year': year,
                             'Method': method, 'Training_Rows': 0,
                             'Max_Selector_Target_Date': pd.NaT,
                             'Selected_Features': '', 'Selected_Count': 0}
                else:
                    transformed, audit = select_context_features(task, cur, year, method)
                    if transformed is None:
                        year_df = task[task['Validation_Year'] == year]
                        sim = pd.DataFrame({
                            'Currency': cur, 'Horizon': h,
                            'Forecast_Origin': year_df['Forecast_Origin'],
                            'Target_Date': year_df['Target_Date'],
                            'Validation_Year': year, 'Actual': year_df['Actual'],
                            'RW_Prediction': year_df['RW'],
                            'BASE_Prediction': year_df['BASE_Prediction'],
                            'Method_Prediction': year_df['BASE_Prediction']})
                    else:
                        sim = simulate_contextual_year(task, year, profile, transformed)
                sim['FS_Method'] = method
                simulations.append(sim)
                fs_audit_rows.append(audit)
        print(f'Finished market-state selectors: {cur} H{h}', flush=True)

fs_simulations = pd.concat(simulations, ignore_index=True)
fs_selection_audit = pd.DataFrame(fs_audit_rows)

In [ ]:
# Select among BASE/MI/L1/SPEARMAN only from already-resolved earlier-year trials.
chosen_parts = []
method_selection_rows = []
for (cur, h), task_sims in fs_simulations.groupby(['Currency', 'Horizon']):
    for year in YEARS:
        start = pd.Timestamp(f'{year}-01-01')
        historical = task_sims[(task_sims['Forecast_Origin'] < start)
                               & (task_sims['Target_Date'] < start)]
        baseline = historical[historical['FS_Method'] == 'BASE']
        base_rmse = rmse(baseline['Actual'], baseline['Method_Prediction'])
        chosen_method = 'BASE'
        chosen_rmse = base_rmse
        # Tie and uncertainty always favor the pre-existing ensemble.
        if year > YEARS[0] and len(baseline) >= FS_MIN_TRAIN and np.isfinite(base_rmse):
            for method in FS_METHODS[1:]:
                sample = historical[historical['FS_Method'] == method]
                if len(sample) != len(baseline):
                    continue
                mr = rmse(sample['Actual'], sample['Method_Prediction'])
                if not np.isfinite(mr) or mr >= base_rmse*(1-FS_IMPROVEMENT_MARGIN):
                    continue
                # Do not accept an apparently good average with a disastrous year.
                stable = True
                for prior_year in sorted(baseline['Validation_Year'].unique()):
                    b = baseline[baseline['Validation_Year'] == prior_year]
                    s = sample[sample['Validation_Year'] == prior_year]
                    br = rmse(b['Actual'], b['Method_Prediction'])
                    sr = rmse(s['Actual'], s['Method_Prediction'])
                    if len(s) != len(b) or not np.isfinite(sr) or (
                        br > 0 and sr > br*(1+FS_MAX_YEARLY_DEGRADATION)):
                        stable = False
                        break
                if stable and mr < chosen_rmse:
                    chosen_method, chosen_rmse = method, mr

        selected = task_sims[(task_sims['Validation_Year'] == year)
                             & (task_sims['FS_Method'] == chosen_method)].copy()
        selected['Selected_FS_Method'] = chosen_method
        selected['Final_Prediction'] = selected['Method_Prediction']
        chosen_parts.append(selected)
        method_selection_rows.append({
            'Currency': cur, 'Horizon': int(h), 'Outer_Year': year,
            'Selected_FS_Method': chosen_method, 'Historical_Resolved_Rows': len(baseline),
            'Historical_BASE_RMSE': base_rmse, 'Historical_Chosen_RMSE': chosen_rmse})

fs_final = pd.concat(chosen_parts, ignore_index=True).sort_values(
    ['Currency', 'Horizon', 'Forecast_Origin']).reset_index(drop=True)
fs_method_selections = pd.DataFrame(method_selection_rows)
print('Selected FS methods across 60 tasks:')
display(fs_method_selections['Selected_FS_Method'].value_counts())

## 21. Honest, comparable evaluation and evidence

The final report compares the feature-selected ensemble with **both** Random
Walk and the exact same-date original Notebook 09 incumbent. We report RMSE,
MAE, Forecast Skill, Theil's U and DM p-values. No claim is made until these
are computed. Feature selection is automatically rejected unless previous
resolved years support it; this does not guarantee future improvements.

In [ ]:
fs_yearly_rows, fs_pooled_rows = [], []
for group_cols, sink in [(['Currency', 'Horizon', 'Validation_Year'], fs_yearly_rows),
                         (['Currency', 'Horizon'], fs_pooled_rows)]:
    for key, g in fs_final.groupby(group_cols):
        cur, h = key[:2]
        fr = rmse(g['Actual'], g['Final_Prediction'])
        rr = rmse(g['Actual'], g['RW_Prediction'])
        br = rmse(g['Actual'], g['BASE_Prediction'])
        _, p_rw = dm_test(g['Actual'], g['Final_Prediction'], g['RW_Prediction'], int(h))
        _, p_base = dm_test(g['Actual'], g['Final_Prediction'], g['BASE_Prediction'], int(h))
        record = {'Currency': cur, 'Horizon': int(h), 'Observations': len(g),
                  'RW_RMSE': rr, 'BASE09_RMSE': br, 'Final_RMSE': fr,
                  'Final_MAE': mae(g['Actual'], g['Final_Prediction']),
                  'Forecast_Skill_vs_RW': skill(fr, rr),
                  'Skill_vs_BASE09': skill(fr, br),
                  'Theils_U': theils_u(fr, rr),
                  'DM_pvalue_vs_RW': p_rw, 'DM_pvalue_vs_BASE09': p_base,
                  'Beats_RW': bool(fr < rr), 'Beats_BASE09': bool(fr < br)}
        if len(key) == 3:
            record['Validation_Year'] = int(key[2])
        sink.append(record)

fs_yearly_results = pd.DataFrame(fs_yearly_rows).sort_values(
    ['Currency', 'Horizon', 'Validation_Year'])
fs_pooled_results = pd.DataFrame(fs_pooled_rows).sort_values(['Currency', 'Horizon'])
display(fs_pooled_results)
print('Numerical RMSE wins vs RW:', int(fs_pooled_results['Beats_RW'].sum()), '/12')
print('Numerical RMSE wins vs original 09:', int(fs_pooled_results['Beats_BASE09'].sum()), '/12')
print('Statistically supported positive wins vs RW:', int((
    fs_pooled_results['Beats_RW'] & (fs_pooled_results['DM_pvalue_vs_RW'] < 0.05)
).sum()), '/12')

In [ ]:
# Mandatory invariants are evaluated, not described as automatic PASS.
key_cols = ['Currency', 'Horizon', 'Forecast_Origin']
safe_target_dates = bool((fs_final['Target_Date'] < HOLDOUT_START).all())
future_only = bool((fs_final['Target_Date'] > fs_final['Forecast_Origin']).all())
unique_keys = not fs_final.duplicated(key_cols).any()
same_keys = (set(map(tuple, fs_final[key_cols].to_numpy()))
             == set(map(tuple, base[key_cols].to_numpy())))
all_finite = bool(np.isfinite(fs_final[
    ['Actual', 'RW_Prediction', 'BASE_Prediction', 'Final_Prediction']
].to_numpy(float)).all())
historical_selectors = all(
    pd.isna(row['Max_Selector_Target_Date'])
    or row['Max_Selector_Target_Date'] < pd.Timestamp(f"{int(row['Outer_Year'])}-01-01")
    for _, row in fs_selection_audit.iterrows()
)
baseline_2019 = bool((fs_final.loc[
    fs_final['Validation_Year'] == 2019, 'Selected_FS_Method'] == 'BASE').all())
no_target_features = all(safe_feature_name(c)
                         for cols in candidate_by_currency.values() for c in cols)
quality_checks = {
    'No final holdout target labels': safe_target_dates,
    'Targets strictly after forecast origin': future_only,
    'Unique currency/horizon/date keys': unique_keys,
    'Exactly same keys as leakage-filtered baseline': same_keys,
    'Finite forecasts and actuals': all_finite,
    'Feature selectors trained before outer year': historical_selectors,
    '2019 pre-selected baseline retained': baseline_2019,
    'No future target columns in features': no_target_features,
}
fs_quality_gate = pd.DataFrame([
    {'Check': k, 'Status': 'PASS' if v else 'FAIL'}
    for k, v in quality_checks.items()
])
display(fs_quality_gate)
if not all(quality_checks.values()):
    raise RuntimeError('Research quality gate failed; outputs were not saved')

# Separate artifact names avoid overwriting the pre-existing Notebook 09 results.
PRED_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
fs_final.to_csv(PRED_DIR/'09FS_safe_predictions_2019_2023.csv', index=False)
fs_pooled_results.to_csv(TABLE_DIR/'09FS_safe_pooled_results.csv', index=False)
fs_yearly_results.to_csv(TABLE_DIR/'09FS_safe_yearly_results.csv', index=False)
fs_method_selections.to_csv(TABLE_DIR/'09FS_safe_method_selections.csv', index=False)
fs_selection_audit.to_csv(TABLE_DIR/'09FS_safe_feature_selection_audit.csv', index=False)
fs_quality_gate.to_csv(TABLE_DIR/'09FS_safe_quality_gate.csv', index=False)
print('Completed. New outputs are prefixed 09FS_safe_ and preserve original files.')
print('IMPORTANT: upstream Script 05/06 forecast provenance needs independent audit.')
print('Do not tune further on the 2024–2025 final holdout.')